# 03. Tracebacks

**Tier:** Intermediate. **Why you'd use this:** A checkout helper calls `tax`, which calls `rate_for`. Something raises. You need to know which call failed first, and you need the five `pdb` commands that get you there when a print is not enough.

**After this lesson you can:** read a traceback from the last line up; name the frames involved; start `breakpoint()` and use `l`, `n`, `s`, `p`, `c`.
About 25 minutes. **Uses:** [Raise](02-raise-lesson.ipynb)


## Quick review

1. What does `raise ... from` store on the new exception?
2. Why return `None` in one helper and raise in another?

<details><summary>Answers</summary>

`__cause__`. Return a stand-in for expected junk. Raise for a broken contract.
</details>


## The idea

A traceback is the stack of frames active when an exception was raised. Read the last line first: the type and message. Then the arrowed line: where it raised. Then walk up to see who called that. `breakpoint()` pauses into `pdb`. The five commands you need are `l` (list), `n` (next line), `s` (step into), `p` expression (print), and `c` (continue).

<details><summary>Explain more</summary>

Each frame is one active call. The bottom frame is where the exception was created. The top of the printed stack is the outermost call. Notebooks and terminals both show that order.
</details>


## See it work

Raise through two helpers and catch the type at the top.


In [1]:
def rate_for(city):
    # 1. Unknown city is a contract failure.
    if city == "":
        raise ValueError("empty city")
    return 0.1


def tax(amount, city):
    # 2. This frame sits above rate_for in the traceback.
    return amount * rate_for(city)


try:
    tax(50, "")
except ValueError as err:
    print(type(err).__name__)
    print(str(err))

ValueError
empty city


## Check yourself

Answer these before you run the cell.

1. Which function name appears nearest the error line in the traceback?
2. What does the cell print for the caught message?


In [2]:
try:
    tax(50, "")
except ValueError as err:
    print(str(err))

empty city


In [3]:
tax(50, "")

ValueError: empty city

<details><summary>Answers</summary>

`rate_for` is nearest the raise. The message is `empty city`. In the uncaught cell, read the last line first, then the arrow in `rate_for`, then the call in `tax`.
</details>


## Why it works this way

Each call pushes a frame. Raising snapshots those frames into a traceback attached to the exception. Handlers can clear or re-raise, but the snapshot explains the path. Reading bottom-up matches how the failure grew: the raise site first, then its callers. Deep dive: [Source to bytecode](../../internals/10-bytecode/01-bytecode-lesson.ipynb).

`breakpoint()` enters the debugger at that line. `l` shows nearby source. `n` runs the next line in this frame. `s` steps into a call. `p` prints an expression in the current frame. `c` continues until the next break or the end. Those five cover almost every teaching debug session. You list, step, inspect, and continue. You do not need the rest of the `pdb` command set to find a bad argument.

The cost of a traceback is walking and formatting frames. That is why hot code avoids raising on the happy path. Keeping `__traceback__` on a caught exception lets logging print the same story later. Clearing it loses the path that made the failure make sense.

When two frames look similar, trust the arrow. The message names the type. The arrow names the line. The frames above name the call path. Fixing a caller without reading the raise site is how you patch the wrong helper and leave the empty-city bug in place.

In notebooks, an uncaught exception stops the cell and shows that same traceback. Re-run the cell after a fix. If you only re-run a later cell, you may be looking at an old definition and an old stack.


In [4]:
try:
    rate_for("")
except ValueError as err:
    print(err.__traceback__ is not None)
    print(type(err).__name__)

True
ValueError


## When to use it

Read the traceback before changing code. Use `breakpoint()` when a print is not enough to see why a value is wrong. Don't catch an exception only to print it and continue, unless you are at a real boundary. Don't scatter `breakpoint()` in committed lesson code except as a short demo you remove.


## Common mistake

Reading the traceback from the top and "fixing" the first file you see. The real raise is at the bottom. The upper frames are callers.


In [5]:
tax(50, "")

ValueError: empty city

Start at `ValueError: empty city`, then the arrow in `rate_for`, then the line in `tax` that called it.


## Try it

Add one more helper, `checkout`, that only calls `tax`. Raise again with an empty city. Which new frame appears above `tax`, and what stays at the bottom?


## Recap

- Read a traceback from the last line, then the arrow, then up the calls.
- Each frame is one active function call.
- `breakpoint()` plus `l`, `n`, `s`, `p`, `c` is enough for most debugging.

**Next:** [practice](03-tracebacks-practice.ipynb)
